# MCP Network Demo (Streamable HTTP Transport) (FOR DEMO)

This notebook demonstrates connecting to a **remote MCP server** over Streamable HTTP transport, using the OpenAI Agents SDK as the client.

We connect to CoinGecko's public MCP server (no API key required).

> **Note:** CoinGecko has redesigned its MCP server to expose just two tools — `search_docs` (find the right SDK method) and `execute` (run a JavaScript snippet against the CoinGecko SDK). The agent uses these to answer any market question, instead of calling per-endpoint tools.

See https://docs.coingecko.com/docs/mcp-server

In [1]:
# Setup
!pip install -q -U openai-agents python-dotenv

In [2]:
import json
import os
import sys
from contextlib import asynccontextmanager

from openai import OpenAI


def load_openai_key():
    """Read the OpenAI API key from Colab Secrets; ask for it only if that fails."""
    try:
        from google.colab import userdata
        return userdata.get("OPENAI_API_KEY")
    except ImportError:
        print("Not running in Colab.")
    except Exception as e:
        print(f"Could not read the Colab secret OPENAI_API_KEY ({type(e).__name__}).")
    from getpass import getpass
    return getpass("Enter your OpenAI API key: ")


os.environ["OPENAI_API_KEY"] = load_openai_key()
openai_client = OpenAI()
MODEL = "gpt-4o-mini"
print("OpenAI client is ready. Model:", MODEL)


OpenAI client is ready. Model: gpt-4o-mini


In [3]:
import json
import logging
from agents.mcp import MCPServerStreamableHttp

logging.getLogger("mcp.client.streamable_http").setLevel(logging.ERROR)

MCP_SERVER_URL = "https://mcp.api.coingecko.com/mcp"
server_params = {"url": MCP_SERVER_URL}

In [4]:
async def list_remote_tools():
    """Connect to remote MCP server and list available tools."""
    async with MCPServerStreamableHttp(params=server_params, client_session_timeout_seconds=60) as server:
        return await server.list_tools()

tools = await list_remote_tools()

print(f"Found {len(tools)} tools:\n")
for tool in tools:
    print(f"- {tool.name}")
    print(f"  {(tool.description or '').splitlines()[0]}\n")

Found 2 tools:

- execute
  Runs JavaScript code to interact with the Coingecko API.

- search_docs
  Search SDK documentation to find methods, parameters, and usage examples for interacting with the API. Use this before writing code when you need to discover the right approach.



In [5]:
import httpx

# CoinGecko's `execute` tool runs a JS snippet against its SDK and returns the result.
JS_CODE = """
async function run(client) {
  return await client.simple.price.get({ vs_currencies: 'usd', ids: 'bitcoin' });
}
"""

async def call_remote_tool(tool_name: str, args: dict):
    """Execute a tool on the remote MCP server. Returns (content_text, error_msg)."""
    try:
        async with MCPServerStreamableHttp(params=server_params, client_session_timeout_seconds=60) as server:
            result = await server.call_tool(tool_name, args)
            content_text = "".join(
                item.text for item in result.content if hasattr(item, "text")
            )
            return content_text, None
    except Exception as e:
        # MCP wraps httpx errors inside an ExceptionGroup
        candidates = e.exceptions if isinstance(e, BaseExceptionGroup) else [e]
        http_errors = [exc for exc in candidates if isinstance(exc, httpx.HTTPStatusError)]
        if http_errors:
            codes = "; ".join(f"HTTP {exc.response.status_code}" for exc in http_errors)
            return None, f"Server error calling {tool_name}: {codes}"
        raise

content, err = await call_remote_tool("execute", {"code": JS_CODE, "intent": "Get current Bitcoin price in USD"})
if err:
    print(f"Error: {err}")
else:
    print(json.dumps(json.loads(content), indent=2))

{
  "result": {
    "bitcoin": {
      "usd": 80860
    }
  }
}


## Agentic Pattern: OpenAI Agents SDK + MCP

The cells above called MCP tools directly. Now we'll let the model decide which tools to call and how to interpret the results.

We hand the `Agent` the MCP server via `mcp_servers=[...]` and `Runner.run` drives the loop internally. For CoinGecko, that means the model can discover the right SDK call via `search_docs`, compose a JavaScript snippet, and run it via `execute`.

## Ask the Agent: BTC vs ETH in the Last 24 Hours

We'll ask the agent to compare BTC and ETH performance. It will autonomously:
- Use `search_docs` to find the right CoinGecko SDK method
- Compose a JavaScript snippet (likely requesting 24h change data)
- Run it via `execute` and reason over the results

In [6]:
from agents import Agent, Runner

SYSTEM_PROMPT = (
    "Use the CoinGecko MCP tools to answer crypto market questions. "
    "If unsure of the right SDK method, call `search_docs` first (language='javascript'), "
    "then call `execute` with a JavaScript snippet. "
    "Cite numeric values from tool results rather than guessing."
)

async def ask_agent_with_mcp(question: str):
    """Run an OpenAI Agent that can call the remote MCP server."""
    print(f"Question: {question}\n")
    async with MCPServerStreamableHttp(params=server_params, client_session_timeout_seconds=60) as server:
        agent = Agent(
            name="Crypto Analyst",
            instructions=SYSTEM_PROMPT,  # SDK's name for the system prompt
            model=MODEL,
            mcp_servers=[server],
        )
        result = await Runner.run(agent, question)
        print(result.final_output)

await ask_agent_with_mcp(
    "Which asset performed better over the last 24 hours, BTC or ETH? Give percentages."
)

Question: Which asset performed better over the last 24 hours, BTC or ETH? Give percentages.

In the last 24 hours:

- **Bitcoin (BTC)** decreased by **-1.19%**
- **Ethereum (ETH)** decreased by **-1.33%**

Therefore, BTC performed better than ETH over this period.
